In [1]:
# 09_real_data.ipynb
# External validation on a real, non-credit, offline dataset (scikit-learn
# breast cancer). A deliberately weak additive scorecard (few features) is fitted
# to obtain a non-degenerate ranking regime, then quantized; the four theorems and
# the closed-form margin density are checked on the real score distribution.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from scipy.stats import gaussian_kde

d = load_breast_cancer(); X = d.data; yb = 1 - d.target        # malignant = 1
aucs = np.array([max(a, 1 - a) for a in [roc_auc_score(yb, X[:, j]) for j in range(X.shape[1])]])
feats = np.argsort(aucs)[:6]                                   # 6 weakest features -> overlap
Xs = X[:, feats]; N, Pf = Xs.shape; B = 5
binidx = np.zeros((N, Pf), int); OH = []
for j in range(Pf):
    ed = np.unique(np.quantile(Xs[:, j], np.linspace(0, 1, B + 1))[1:-1])
    bi = np.clip(np.digitize(Xs[:, j], ed), 0, len(ed)); binidx[:, j] = bi
    oh = np.zeros((N, len(ed) + 1)); oh[np.arange(N), bi] = 1; OH.append(oh)
Z = np.hstack(OH); lr = LogisticRegression(C=1.0, max_iter=3000).fit(Z, yb)
off = np.cumsum([0] + [o.shape[1] for o in OH]); co = lr.coef_[0]
Cr = np.stack([co[off[j] + binidx[:, j]] for j in range(Pf)], axis=1); b0r = float(lr.intercept_[0]); fr = b0r + Cr.sum(1)
AUC = roc_auc_score(yb, fr)
def fqr(D): return b0r + (D * np.round(Cr / D)).sum(1)
def migration(fc, s, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1]); return float(np.mean(np.digitize(fc, q) != np.digitize(s, q)))
def reversal(fc, s, seed=0, npair=200000):
    rr = np.random.default_rng(seed); a = rr.choice(np.where(yb == 1)[0], npair); b = rr.choice(np.where(yb == 0)[0], npair)
    return float(np.mean((np.sign(fc[a] - fc[b]) > 0) & (np.sign(s[a] - s[b]) <= 0)))

Deltas = np.array([0.05, 0.1, 0.2, 0.4]); rows = []
for D in Deltas:
    s = fqr(D); e = s - fr
    rows.append(dict(Delta=D, std_e=e.std(), typical=D * np.sqrt(Pf / 12),
                     auc_loss=AUC - roc_auc_score(yb, s), migration_K10=migration(fr, s, 10)))
tab = pd.DataFrame(rows); tab.to_csv("../results/tables/09_real_data.csv", index=False)
fp, fn = fr[yb == 1], fr[yb == 0]; grid = np.linspace(fr.min(), fr.max(), 300); h = grid[1] - grid[0]
phi = float(np.sum(gaussian_kde(fp)(grid) * gaussian_kde(fn)(grid)) * h)
r = np.random.default_rng(0); dd = fp[r.integers(0, len(fp), 300000)] - fn[r.integers(0, len(fn), 300000)]
phi_d = float(np.mean(np.abs(dd) < 0.2) / 0.4)
D = 0.2; s = fqr(D); R = reversal(fr, s); Ks = [5, 7, 10, 20]; ratios = [migration(fr, s, K) / max(R, 1e-6) for K in Ks]
print("breast_cancer weak scorecard: N=%d P=%d AUC=%.3f base rate=%.3f" % (N, Pf, AUC, yb.mean()))
print(tab.round(4).to_string(index=False))
print("phi_m(0): closed=%.4f direct=%.4f" % (phi, phi_d))
print("separation ratio M/R by K (Delta=0.2):", list(zip(Ks, [round(x, 1) for x in ratios])))

fig, ax = plt.subplots(2, 2, figsize=(9, 7))
ax[0, 0].plot(tab.Delta, tab.std_e, "-o", color="0.0", label=r"observed $\mathrm{std}(e)$")
ax[0, 0].plot(tab.Delta, tab.typical, ":^", color="0.5", label=r"$\Delta\sqrt{P/12}$")
ax[0, 0].set_xlabel(r"$\Delta$"); ax[0, 0].set_ylabel("perturbation scale"); ax[0, 0].legend(fontsize=7)
axb = ax[0, 1]
l1 = axb.plot(tab.Delta, tab.migration_K10, "-o", color="0.0", label="migration (K=10)")
axb.set_xlabel(r"$\Delta$"); axb.set_ylabel("migration rate")
ax2 = axb.twinx(); l2 = ax2.plot(tab.Delta, tab.auc_loss, "--s", color="0.55", label="AUC loss"); ax2.set_ylabel("AUC loss"); ax2.grid(False)
axb.legend(handles=l1 + l2, fontsize=7, loc="upper left")
ax[1, 0].plot(grid, gaussian_kde(fp)(grid), "-", color="0.0", label=r"$\rho_+$")
ax[1, 0].plot(grid, gaussian_kde(fn)(grid), "--", color="0.45", label=r"$\rho_-$")
ax[1, 0].fill_between(grid, gaussian_kde(fp)(grid) * gaussian_kde(fn)(grid), color="0.8", label=r"$\rho_+\rho_-$")
ax[1, 0].set_xlabel("score"); ax[1, 0].set_ylabel("density"); ax[1, 0].legend(fontsize=7, title=r"$\phi_m(0)$=%.3f (%.3f)" % (phi, phi_d))
ax[1, 1].plot(Ks, ratios, "-o", color="0.0")
ax[1, 1].set_xlabel("number of bands K"); ax[1, 1].set_ylabel(r"M/R (at $\Delta=0.2$)")
fig.tight_layout(); _save(fig, "09_real_data"); plt.close(fig)

breast_cancer weak scorecard: N=569 P=6 AUC=0.836 base rate=0.373
 Delta  std_e  typical  auc_loss  migration_K10
  0.05 0.0284   0.0354    0.0002         0.0633
  0.10 0.0697   0.0707    0.0020         0.1687
  0.20 0.1544   0.1414    0.0014         0.2566
  0.40 0.2006   0.2828    0.0038         0.4815
phi_m(0): closed=0.1534 direct=0.1549
separation ratio M/R by K (Delta=0.2): [(5, 6.5), (7, 8.9), (10, 13.4), (20, 26.5)]
